# Homework 6 on trees

# Data preparation

In [16]:
!pip install xgboost

import pandas as pd
import numpy as np
import xgboost as xgb
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from matplotlib import pyplot as plt
%matplotlib inline


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
#Download the dataset from the given URL 
dataset_url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
!wget $dataset_url

--2026-11-02 19:04:33--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.008s  

2026-11-02 19:04:33 (79.7 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



In [4]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [5]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [6]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [9]:
#determine the unique values in the "horsepower" column
df["horsepower"].unique()
df["acceleration"].unique()

array([ nan, 17.3, 18.7, 17.5, 18.8, 21.7, 17.4, 19.5, 17.7, 16.7, 20.7,
       20.2, 18. , 20.1, 17.9, 15.8, 19.6, 19.4, 17.1, 20.4, 20.6, 18.6,
       18.2, 16.8, 20.5, 21.3, 21.1, 19. , 18.1, 18.3, 20.9, 19.1, 19.3,
       16.1, 19.8, 19.2, 17.8, 17.2, 20. , 19.7, 18.9, 21.2, 18.5, 17.6,
       21.5, 19.9, 17. , 18.4, 21. , 16.5, 16. , 20.3, 20.8, 22.2, 16.4,
       16.6, 21.6, 21.4, 15.5, 22.6, 16.3, 16.9, 15.4, 22.8, 15.6, 15. ,
       21.9, 21.8, 15.1, 15.7, 15.9, 22. , 15.3, 22.5, 22.1, 16.2, 15.2,
       22.9, 14.6, 22.4, 23.4, 22.3, 22.7, 14.7, 23. , 14.9, 14.8, 23.3,
       14.4, 23.2, 14.3, 14. , 23.8, 14.5])

In [11]:
#Filling missing values in the "horsepower" and "acceleration" columns
df["horsepower"]= df["horsepower"].fillna(0.0)
df["acceleration"]= df["acceleration"].fillna(0.0)

In [17]:
df.isnull().sum()
df.shape

(10000, 11)

# Data validation setting up

In [19]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [ ]:
#Reset the index of the dataframes to ensure they are sequential and start from 0
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

#Get the target variable (fuel_efficiency_mpg) and remove it from the training, validation, and test sets
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

#Transform the data into a format suitable for machine learning models using DictVectorizer
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

X_train.shape, X_val.shape 

((6000, 16), (2000, 16))

# Q1 Train decision tree with max-depth 1

In [ ]:
#used DecisionTreeRegressor to fit the training data and print the decision tree structure
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



# Q2 Train a random forest regressor with n_estimators=10

In [27]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [30]:
rf = RandomForestRegressor(n_estimators=10, random_state=1,n_jobs=-1)
rf.fit(X_train, y_train)

score = rmse(y_val, rf.predict(X_val))
score, round(score, 3)

(np.float64(1.8332197358745623), np.float64(1.833))

# Q3 Experiment with the `n_estimators` parameter

In [31]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1,n_jobs=-1)
    rf.fit(X_train, y_train)
    score = rmse(y_val, rf.predict(X_val))
    print(f"n_estimators: {n}, RMSE: {score:.3f}")

n_estimators: 10, RMSE: 1.833
n_estimators: 50, RMSE: 1.774
n_estimators: 100, RMSE: 1.770
n_estimators: 150, RMSE: 1.769


# Q4 Find the best Max-depth around estimators

In [32]:
for max_depth in [10, 15, 20, 25]:
    scores = []
    for n_estimators in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=1,n_jobs=-1)
        rf.fit(X_train, y_train)
        score = rmse(y_val, rf.predict(X_val))
        scores.append(score)
    print(f"max_depth: {max_depth}, Mean RMSE: {np.mean(scores):.3f}")

max_depth: 10, Mean RMSE: 1.754
max_depth: 15, Mean RMSE: 1.784
max_depth: 20, Mean RMSE: 1.787
max_depth: 25, Mean RMSE: 1.786


# Q5 Feature importance
* Train the model with these parameters:
  * `n_estimators=10`,
  * `max_depth=20`,
  * `random_state=1`,
  * `n_jobs=-1` (optional)
* Get the feature importance information from this model

In [ ]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1,n_jobs=-1)
rf.fit(X_train, y_train)

#most important features between "vehicle_weight" and "horsepower" , "acceleration" and "engine_displacement"
for feature, importance in zip(dv.get_feature_names_out(), rf.feature_importances_):
    if feature in ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]:
        print(f"{feature}: {importance:.3f}")


acceleration: 0.055
engine_displacement: 0.061
horsepower: 0.078
vehicle_weight: 0.203


# Q6 XGBoost 

In [37]:
features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]



In [38]:
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=25)

    print(f'eta={eta}:  train_rmse={rmse(y_train, model.predict(dtrain)):.6f}'
          f'   val_rmse={rmse(y_val, model.predict(dval)):.6f}')
    print()

[0]	train-rmse:2.40653	val-rmse:2.46782
[25]	train-rmse:1.35837	val-rmse:1.74359
[50]	train-rmse:1.18645	val-rmse:1.77517
[75]	train-rmse:1.02727	val-rmse:1.80682
[99]	train-rmse:0.90106	val-rmse:1.82646
eta=0.3:  train_rmse=0.901056   val_rmse=1.826458

[0]	train-rmse:2.74739	val-rmse:2.78635
[25]	train-rmse:1.50918	val-rmse:1.72751
[50]	train-rmse:1.40368	val-rmse:1.71358
[75]	train-rmse:1.35099	val-rmse:1.72057
[99]	train-rmse:1.31192	val-rmse:1.72481
eta=0.1:  train_rmse=1.311918   val_rmse=1.724810

